# Data preparation – CNB interest rates

Merging three CNB exports (new business, deposits, repo rate) into a single monthly table for analysing the relationship between the repo rate and mortgages. Output: `cnb_merged.csv`.

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

## Loans – new business

In [ ]:
# ARAD export: semicolon separator, comma as the decimal mark
df = pd.read_csv("/content/drive/MyDrive/loans_new_business.csv",
                 sep=";", decimal=",", encoding="utf-8-sig")
df.shape

In [ ]:
# CNB columns have long names and many sub-categories; pick the main indicator
# and the 'must_not' list filters out the breakdowns (fixations, refinanced, building societies...)
def find_columns(must, must_not):
    return [c for c in df.columns
            if all(m in c for m in must) and not any(n in c for n in must_not)]

must_not = ["Refinancované", "Floating", "Nad", "Čisté", "Ostatní", "Stavební spořitelny",
            "Banky bez", "Čtvrtletní", "RPSN", "Velké banky", "EUR", "Kontokorent",
            "revolving", "Do ", "Od ", "a do"]

mortgage_rate   = find_columns(["Měsíční", "Úroková sazba", "Hypoteční na bytové nemovitosti", "Banky včetně stavebních spořitelen", "(%)"], must_not)
mortgage_volume = find_columns(["Měsíční", "Objem", "Hypoteční na bytové nemovitosti", "Banky včetně stavebních spořitelen", "(mil. CZK)"], must_not)
consumer_rate   = find_columns(["Měsíční", "Úroková sazba", "Domácnosti - obyvatelstvo", "Spotřebitelské (bez KTK", "Banky včetně stavebních spořitelen", "(%)"], must_not)
consumer_volume = find_columns(["Měsíční", "Objem", "Domácnosti - obyvatelstvo", "Spotřebitelské (bez KTK", "Banky včetně stavebních spořitelen", "(mil. CZK)"], must_not)

# each filter should return exactly one column
print(len(mortgage_rate), len(mortgage_volume), len(consumer_rate), len(consumer_volume))

In [ ]:
loans = df[[df.columns[0], mortgage_rate[0], mortgage_volume[0], consumer_rate[0], consumer_volume[0]]].copy()
loans.columns = ["period", "mortgage_rate", "mortgage_volume", "consumer_rate", "consumer_volume"]
loans["period"] = pd.to_datetime(loans["period"])
loans.head()

## Household deposits

In [ ]:
deposits = pd.read_csv("/content/drive/MyDrive/deposits.csv",
                       sep=";", decimal=",", encoding="utf-8-sig")

# this table is transposed (dates in columns), so select the households row
# and flip it with .T so the dates end up in rows like in the other tables
row = deposits[deposits["Ukazatel"].str.startswith("6. Domácnosti")]
dep = row.set_index("Ukazatel").T.reset_index()
dep.columns = ["period", "household_deposits"]

# the export has a trailing empty column that turns into an invalid row after transposing
dep = dep[~dep["period"].astype(str).str.startswith("Unnamed")]
dep["period"] = pd.to_datetime(dep["period"])
dep.head()

## Repo rate (CNB 2W repo)

In [ ]:
repo = pd.read_csv("/content/drive/MyDrive/repo_rate.txt", sep="|", decimal=",")
repo["PLATNA_OD"] = pd.to_datetime(repo["PLATNA_OD"], format="%Y%m%d")
repo = repo.sort_values("PLATNA_OD").rename(columns={"CNB_REPO_SAZBA_V_%": "repo_rate"})

In [ ]:
# the repo rate changes in steps (only on the day of the change), so assign each month
# the last valid rate (backward merge_asof)
repo_monthly = pd.merge_asof(
    loans[["period"]].sort_values("period"),
    repo,
    left_on="period",
    right_on="PLATNA_OD",
    direction="backward"
)[["period", "repo_rate"]]

repo_monthly.tail()

## Merge and export

In [ ]:
merged = loans.merge(repo_monthly, on="period").merge(dep, on="period")
merged = merged.sort_values("period")

# CNB only started collecting loan statistics in 2004, so drop the older months
# (where only repo and deposits exist) to keep the period complete
merged = merged.dropna()

print(merged.shape, "| from", merged["period"].min().date(), "to", merged["period"].max().date())
merged.head()

In [ ]:
merged.to_csv("cnb_merged.csv", index=False, encoding="utf-8-sig")